# Desafio A - Compras públicas e diferenças regionais 
---

## Problema
- A distribuiçao das contrataçoes públicas apresenta difereças a entre as Unidade da Federação?

## Estrategia de resolução 

- Vamos criar um mapa de calor com um filtro de anos, de 2020 a 2026. Será possível visualizar as contratações de cada ano e também o total acumulado do período.

O mapa vai mostrar todos os países e a quantidade de contratações realizadas. Quanto mais forte a cor, maior o número de contratações; quanto mais fraca, menor a quantidade. Assim, fica fácil identificar quais países mais contratam ao longo dos anos.

## Perguntas 
- Quantidade de contratação aentre os estados
- Estados Com mais contratações 
- Estados com menos contratações 
- Estados com maior concetração em diferentes modalidades 
- Os vaalores gastos por cada estaçao por ano analisado + somativo geral nacional 

## Análises 
- Frequencias 
- Mediana 
- Média 
- Variação 
- Maior 
- Minimo 

## Filtros utilizados da API
- Consultar Contratacoes 
- Consultar Itens
- Consualtar resultados itens contratacoes

## Bibliotecas

Utilizaremos principalmente:

- `requests` para realizar as requisições HTTP;
- `pandas` para organizar e analisar os dados;
- `matplotlib` para visualizações simples.

In [ ]:
#%pip install requests pandas matplotlib -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

## URLs ultilizadas:

A API do Compras.gov.br possui diferentes módulos.

Alguns exemplos:

```text
Catálogo de materiais
Catálogo de serviços
Pesquisa de preços
Contratações
Atas de Registro de Preços
Contratos
Fornecedores
```

Nesse projeto ultilizaremos o modulo de contratações, fazendo uso dos seguintes endpoints:

```text
GET /modulo-contratacoes/1_consultarContratacoes_PNCP_14133
GET //modulo-contratacoes/2_consultarItensContratacoes_PNCP_14133
GET //modulo-contratacoes/3_consultarResultadoItensContratacoes_PNCP_14133
```

Esse serviço consulta contratações realizadas no contexto da Lei nº 14.133/2021. Esta Lei estabelece normas gerais de licitação e contratação para as Administrações Públicas diretas, autárquicas e fundacionais da União, dos Estados, do Distrito Federal e dos Municípios

A URL base será:

In [3]:
base_url = "https://dadosabertos.compras.gov.br"

endpoint_contratacoes = "/modulo-contratacoes/1_consultarContratacoes_PNCP_14133"
endpoint_itens = "/modulo-contratacoes/2_consultarItensContratacoes_PNCP_14133"
endpoint_resultados_itens = "/modulo-contratacoes/3_consultarResultadoItensContratacoes_PNCP_14133"


## Parâmetros da consulta

O endpoint permite utilizar parâmetros para restringir a consulta.

Nesta atividade utilizaremos:

- `anos`: anos dos registros;
- `tamanhoPagina`: número de registros por página;
- `maximoPaginas`: maxímo de registros por página;
- `codigoModalidade`: código da modalidade;



In [4]:
anos = list(range(2020, 2027))          
modalidade = 6                         
tamanho_paginas = 50                 
max_paginas_ano = 20              

parametros = {
    "ano": anos,
    "tamanhoPagina": tamanho_paginas,
    "maximoPaginas": max_paginas_ano,
    "codigoModalidade": modalidade,
}

parametros

{'ano': [2020, 2021, 2022, 2023, 2024, 2025, 2026],
 'tamanhoPagina': 50,
 'maximoPaginas': 20,
 'codigoModalidade': 6}

## Consulta simples dos dados

Antes da coleta completa (2020–2026), fizemos uma consulta pequena para verificar:

- o endpoint estava disponível?
- os campos necessários (UF, valor, modalidade) estão presentes?
- o `codigoModalidade` escolhido retorna registros?

In [8]:
def extrair_informacoes(dados):
    """Extrai os registros da resposta."""
    return dados["resultado"]

params_dados = {
    "pagina": 1,
    "tamanhoPagina": 10,
    "dataPublicacaoPncpInicial": "2025-05-01",
    "dataPublicacaoPncpFinal": "2025-05-15",
    "codigoModalidade": modalidade,
}

resposta = requests.get(base_url + endpoint_contratacoes, params=params_dados)

print("Status HTTP:", resposta.status_code)
print("URL consultada:", resposta.url)

registros = extrair_informacoes(resposta.json())
df_simples = pd.json_normalize(registros)

print("Registros encontrados:", len(df_simples))
df_simples.head()

Status HTTP: 200
URL consultada: https://dadosabertos.compras.gov.br/modulo-contratacoes/1_consultarContratacoes_PNCP_14133?pagina=1&tamanhoPagina=10&dataPublicacaoPncpInicial=2025-05-01&dataPublicacaoPncpFinal=2025-05-15&codigoModalidade=6
Registros encontrados: 10


,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,tipoInstrumentoConvocatorioNome,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida
0,78251006900072025,00394502000144-1-003952/2025,2025,3952,00394502000144,None,46041,COMANDO DA MARINHA,None,F,...,Aviso de Contratação Direta,Dispensa Com Disputa,34103.12,NaN,2025-05-01T08:48:39,2025-05-01T08:48:39,2025-05-01T08:48:39,2025-05-01T08:48:36,2025-05-08T07:59:59,False
1,92826706900152025,14846532000159-1-000021/2025,2025,21,14846532000159,None,67124,CONSELHO DE ARQUITETURA E URBANISMO DO AMAPA,None,F,...,Aviso de Contratação Direta,Dispensa Com Disputa,4100.00,1500.0,2025-05-01T11:08:48,2025-05-01T11:08:48,2025-05-01T11:08:48,2025-05-01T11:08:47,2025-05-08T07:59:59,False
2,78882006900212025,00394502000144-1-003953/2025,2025,3953,00394502000144,None,46041,COMANDO DA MARINHA,None,F,...,Aviso de Contratação Direta,Dispensa Com Disputa,7451.86,NaN,2025-05-01T11:24:21,2025-05-01T11:24:21,2025-05-01T11:24:21,2025-05-01T11:24:19,2025-05-08T07:59:59,False
3,78790006900502025,00394502000144-1-003954/2025,2025,3954,00394502000144,None,46041,COMANDO DA MARINHA,None,F,...,Aviso de Contratação Direta,Dispensa Com Disputa,4667.26,4275.7,2025-05-01T12:12:43,2025-05-01T12:12:43,2025-05-01T12:12:43,2025-05-01T12:12:42,2025-05-08T07:59:59,False
4,16000606900072025,00394452000103-1-009791/2025,2025,9791,00394452000103,None,44611,COMANDO DO EXERCITO,None,F,...,Aviso de Contratação Direta,Dispensa Com Disputa,76274.86,63987.0,2025-05-01T12:17:43,2025-05-01T12:17:43,2025-05-01T12:17:43,2025-05-01T12:17:42,2025-05-13T08:59:59,False


In [11]:
colunas_interesse = [
    "idCompra",
    "numeroCompra",
    "anoCompraPncp",
    "modalidadeNome",
    "modalidadeIdPncp",
    "unidadeOrgaoNomeUnidade",
    "orgaoEntidadeRazaoSocial",
    "objetoCompra",
    "valorTotalEstimado",
    "valorTotalHomologado",
    "dataPublicacaoPncp",
    "srp",
    "unidadeOrgaoUfSigla"
]

colunas_existentes = []

for coluna in colunas_interesse:
    if coluna in df_simples.columns:
        colunas_existentes.append(coluna)

print("Colunas retornadas pela API:")
print(df_simples.columns.tolist())

print("\nColunas de interesse encontradas:")
print(colunas_existentes)

print("\nColuna da UF:", "unidadeOrgaoUfSigla")

Colunas retornadas pela API:
['idCompra', 'numeroControlePNCP', 'anoCompraPncp', 'sequencialCompraPncp', 'orgaoEntidadeCnpj', 'orgaoSubrogadoCnpj', 'codigoOrgao', 'orgaoEntidadeRazaoSocial', 'orgaoSubrogadoRazaoSocial', 'orgaoEntidadeEsferaId', 'orgaoSubrogadoEsferaId', 'orgaoEntidadePoderId', 'orgaoSubrogadoPoderId', 'unidadeOrgaoCodigoUnidade', 'unidadeSubrogadaCodigoUnidade', 'unidadeOrgaoNomeUnidade', 'unidadeSubrogadaNomeUnidade', 'unidadeOrgaoUfSigla', 'unidadeSubrogadaUfSigla', 'unidadeOrgaoMunicipioNome', 'unidadeSubrogadaMunicipioNome', 'unidadeOrgaoCodigoIbge', 'unidadeSubrogadaCodigoIbge', 'numeroCompra', 'modalidadeIdPncp', 'codigoModalidade', 'modalidadeNome', 'srp', 'modoDisputaIdPncp', 'codigoModoDisputa', 'amparoLegalCodigoPncp', 'amparoLegalNome', 'amparoLegalDescricao', 'informacaoComplementar', 'processo', 'objetoCompra', 'existeResultado', 'orcamentoSigilosoCodigo', 'orcamentoSigilosoDescricao', 'situacaoCompraIdPncp', 'situacaoCompraNomePncp', 'tipoInstrumentoConvo

## 9. Coleta dos dados

Essa função irá coletar, dados de um determinado ano, de acordo com o `max`, seguindo as boas práticas da aula: paginação controlada, `timeout`, interrupção quando a página vier vazia.

In [12]:
def coletar_ano(ano):
    data_inicial = f"{ano}-01-01"
    data_final = f"{ano}-12-31"

    todos_registros = []

    for pagina in range(1, max_paginas_ano + 1):

        params = {
            "pagina": pagina,
            "tamanhoPagina": tamanho_paginas,
            "dataPublicacaoPncpInicial": data_inicial,
            "dataPublicacaoPncpFinal": data_final,
            "codigoModalidade": modalidade,
        }

        resposta = requests.get(
            base_url + endpoint_contratacoes,
            params=params,
            timeout=60
        )

        print(f"Ano {ano} - Página {pagina} - Status: {resposta.status_code}")

        if resposta.status_code != 200:
            break

        registros = extrair_informacoes(resposta.json())

        print(f"Registros encontrados: {len(registros)}")

        if len(registros) == 0:
            break

        todos_registros.extend(registros)

        if len(registros) < tamanho_paginas:
            break

        time.sleep(0.3)

    df_ano = pd.json_normalize(todos_registros)

    if not df_ano.empty:
        df_ano["anoConsulta"] = ano

    return df_ano